# ENS-001 — усреднение вероятностей пяти моделей

Цель опыта — проверить **разнообразный** ансамбль: логистическая регрессия, Random Forest, XGBoost, CatBoost и MLP-024.

Все участники получают одни и те же пять outer CV folds. На каждой строке мы усредняем именно вероятности класса `Survived=1`, а затем применяем порог `0.5`. Это называется **soft voting**.


## 1. Подготовка окружения

Эта ячейка только находит корень проекта и импортирует зафиксированный модуль ENS-001.


In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").exists() and (candidate / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog
from ml_project import config as project_config
import ml_project.ensemble_experiment as ensemble_tools

ensemble_tools = importlib.reload(ensemble_tools)
ACTIVE_SPEC = ensemble_tools.SPEC
print("Project root:", PROJECT_ROOT)
print("Experiment:", ACTIVE_SPEC.experiment_id)


## 2. Проверка состава до обучения

Здесь видно, откуда взят каждый участник. Параметры деревьев и бустингов скопированы в модуль ENS-001, поэтому последующее редактирование screening-конфига не изменит этот эксперимент.


In [ ]:
display(
    __import__("pandas").DataFrame(
        [
            {"model": model, "source": source, "parameters": ensemble_tools.MODEL_PARAMETERS.get(model, "versioned module")}
            for model in ACTIVE_SPEC.members
            for source in [ensemble_tools.MODEL_SOURCES[model]]
        ]
    )
)


## 3. Загрузка train

Загружается исходный `train.csv`. Preprocessing и статистики признаков обучаются внутри каждого train fold соответствующей модели.


In [ ]:
catalog = DataCatalog(PROJECT_ROOT, project_config.RAW_DIR, project_config.DATASETS)
catalog.validate()
train = catalog.load(project_config.TRAIN_DATASET)
print("Train shape:", train.shape)
display(train.head(3))


## 4. Общий OOF-прогон

Это главная и самая долгая ячейка. Она обучает 5 моделей × 5 folds. `n_jobs=1` внутри тяжёлых моделей защищает память от множества параллельных OpenBLAS-процессов.


In [ ]:
result = ensemble_tools.run_probability_ensemble(PROJECT_ROOT, train, spec=ACTIVE_SPEC)
display(result.summary.round(4))


## 5. Что дал ансамбль

Сравниваем среднее вероятностей с каждым участником на тех же folds. Главный вопрос: выше ли `accuracy_mean` ансамбля, чем у лучшей одиночной модели этого запуска?


In [ ]:
ensemble_row = result.summary.query("model == 'ensemble_mean'").iloc[0]
best_single = result.summary.query("model != 'ensemble_mean'").iloc[0]
print(f"Ensemble: {ensemble_row.accuracy_mean:.4f} ± {ensemble_row.accuracy_std:.4f}")
print(f"Best single: {best_single.model} — {best_single.accuracy_mean:.4f} ± {best_single.accuracy_std:.4f}")
print(f"Delta: {ensemble_row.accuracy_mean - best_single.accuracy_mean:+.4f}")
display(result.fold_metrics.pivot(index="fold", columns="model", values="accuracy").round(4))


## 6. Действительно ли модели разные?

- **Корреляция вероятностей** около `1.0` означает очень похожую уверенность моделей.
- **Доля разных классов** показывает, как часто две модели стоят по разные стороны порога `0.5`.
- **Пересечение ошибок** — Jaccard: `1.0` означает одинаковый набор ошибочных пассажиров, `0.0` — ошибки не пересекаются.

Для ансамбля полезно, когда сильные модели не полностью повторяют ошибки друг друга.


In [ ]:
print("Корреляции OOF-вероятностей")
display(result.probability_correlations.round(3))
print("Доля несовпадающих предсказанных классов")
display(result.prediction_disagreements.round(3))
print("Jaccard-пересечение ошибок")
display(result.error_overlap.round(3))


## 7. Пассажиры, которых ансамбль исправил или испортил

Таблица помогает понять механику результата, а не смотреть только на одно среднее число.


In [ ]:
oof = result.oof_predictions.copy()
best_name = str(best_single.model)
oof["ensemble_correct"] = oof["prediction_ensemble_mean"].eq(oof["target"])
oof["best_single_correct"] = oof[f"prediction_{best_name}"].eq(oof["target"])

fixed = oof[oof["ensemble_correct"] & ~oof["best_single_correct"]]
broken = oof[~oof["ensemble_correct"] & oof["best_single_correct"]]
print("Исправлено относительно лучшей одиночной модели:", len(fixed))
print("Испорчено относительно лучшей одиночной модели:", len(broken))
display(fixed.head(15))
display(broken.head(15))


## 8. Сохранение и карточка Obsidian

Запускайте после анализа результата. Ячейка сохраняет OOF-вероятности, fold-метрики, таблицы разнообразия и обновляет карточку `ENS-001`. Решение останется `pending`, пока вы сами не выберете `adopt` или `reject`.


In [ ]:
run_dir, note_path = ensemble_tools.save_ensemble_run(PROJECT_ROOT, result, spec=ACTIVE_SPEC)
print("Artifacts:", run_dir.relative_to(PROJECT_ROOT))
print("Obsidian card:", note_path.relative_to(PROJECT_ROOT))
